In [ ]:
!pip install -q deepagents langchain langgraph langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.8/337.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.1/812.1 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 17.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [ ]:
from typing import TypedDict, Annotated
import operator

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

In [ ]:
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key="",
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
class GameState(TypedDict):
    game_idea: str

    story: str
    characters: str
    gameplay: str

    aggregated: str

    votes: Annotated[list, operator.add]
    feedback: Annotated[list, operator.add]

    round: int

In [ ]:
story_agent = create_deep_agent(
    model=model,
    system_prompt="""
You are the Story Agent for a game development team.

Your job is to develop and improve the STORY of the proposed game.

Focus on:
- Setting
- Main conflict
- Plot
- Goals
- Interesting twists

Keep the story coherent with the original game idea.

If feedback from previous voting is provided, use it to improve the story.

Return only your story analysis.
"""
)

In [ ]:
character_agent = create_deep_agent(
    model=model,
    system_prompt="""
You are the Character Agent for a game development team.

Your job is to develop and improve the CHARACTERS of the proposed game.

Focus on:
- Main protagonist
- Important supporting characters
- Enemies
- Character motivations
- Character relationships

Make sure the characters fit the game's story.

If feedback from previous voting is provided, use it to improve the characters.

Return only your character analysis.
"""
)

In [ ]:
gameplay_agent = create_deep_agent(
    model=model,
    system_prompt="""
You are the Gameplay Agent for a game development team.

Your job is to develop and improve the GAMEPLAY.

Focus on:
- Core gameplay loop
- Player actions
- Game mechanics
- Progression
- Challenges
- How the player interacts with the world

Make sure the gameplay fits the story and characters.

If feedback from previous voting is provided, use it to improve the gameplay.

Return only your gameplay analysis.
"""
)

In [ ]:
aggregator_agent = create_deep_agent(
    model=model,
    system_prompt="""
You are the Game Concept Aggregator.

You receive:
- The original game idea
- Story analysis
- Character analysis
- Gameplay analysis
- Feedback from previous voting rounds

Your job is to combine everything into ONE coherent game concept.

Make sure:
- The story fits the characters
- The characters fit the gameplay
- The gameplay supports the story
- The final concept is realistic and coherent

Return a clear game concept containing:

1. Game Title
2. Story
3. Characters
4. Gameplay
5. Overall Concept
"""
)

In [ ]:
voter_1 = create_deep_agent(
    model=model,
    system_prompt="""
You are Game Reviewer 1.

Review the proposed game concept.

Decide whether you would approve the concept.

Return EXACTLY this format:

VOTE: YES
REASON: <short reason>

or

VOTE: NO
REASON: <short explanation of what should be improved>

Be critical but constructive.
"""
)

In [ ]:
voter_2 = create_deep_agent(
    model=model,
    system_prompt="""
You are Game Reviewer 2.

Review the proposed game concept.

Decide whether you would approve the concept.

Return EXACTLY this format:

VOTE: YES
REASON: <short reason>

or

VOTE: NO
REASON: <short explanation of what should be improved>

Be critical but constructive.
"""
)

In [ ]:
voter_3 = create_deep_agent(
    model=model,
    system_prompt="""
You are Game Reviewer 3.

Review the proposed game concept.

Decide whether you would approve the concept.

Return EXACTLY this format:

VOTE: YES
REASON: <short reason>

or

VOTE: NO
REASON: <short explanation of what should be improved>

Be critical but constructive.
"""
)

In [ ]:
def get_agent_response(agent, prompt):
    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    })

    return result["messages"][-1].content

In [ ]:
def orchestrator(state: GameState):

    round_number = state["round"]

    print(f"\n========== ROUND {round_number} ==========")
    print("\nORCHESTRATOR")
    print("Starting game concept discussion...")

    if state["feedback"]:

        print("\nPrevious voting feedback:")
        for feedback in state["feedback"]:
            print("-", feedback)

    else:
        print("\nThis is the first discussion round.")

    return {
        "round": round_number
    }

In [ ]:
def story_node(state: GameState):

    feedback_text = "\n".join(state["feedback"])

    prompt = f"""
We are developing this game:

GAME IDEA:
{state["game_idea"]}

This is discussion round {state["round"]}.

Previous reviewer feedback:
{feedback_text if feedback_text else "No previous feedback."}

Develop the story of the game.

Focus specifically on:
- Setting
- Main conflict
- Plot
- Player's goal
- Interesting events

Improve the story if previous feedback exists.
"""

    response = get_agent_response(story_agent, prompt)

    print("\n--- STORY AGENT ---")
    print(response)

    return {
        "story": response
    }

In [ ]:
def character_node(state: GameState):

    feedback_text = "\n".join(state["feedback"])

    prompt = f"""
We are developing this game:

GAME IDEA:
{state["game_idea"]}

Current story:

{state["story"]}

This is discussion round {state["round"]}.

Previous reviewer feedback:
{feedback_text if feedback_text else "No previous feedback."}

Develop the characters.

Focus on:
- Protagonist
- Supporting characters
- Enemies
- Motivations
- Relationships

Make sure the characters fit the story.
"""

    response = get_agent_response(character_agent, prompt)

    print("\n--- CHARACTER AGENT ---")
    print(response)

    return {
        "characters": response
    }

In [ ]:
def gameplay_node(state: GameState):

    feedback_text = "\n".join(state["feedback"])

    prompt = f"""
We are developing this game:

GAME IDEA:
{state["game_idea"]}

STORY:
{state["story"]}

CHARACTERS:
{state["characters"]}

This is discussion round {state["round"]}.

Previous reviewer feedback:
{feedback_text if feedback_text else "No previous feedback."}

Develop the gameplay.

Focus on:
- Core gameplay loop
- Player actions
- Mechanics
- Challenges
- Progression
- Player interaction

Make sure the gameplay fits the story and characters.
"""

    response = get_agent_response(gameplay_agent, prompt)

    print("\n--- GAMEPLAY AGENT ---")
    print(response)

    return {
        "gameplay": response
    }

In [ ]:
def aggregator(state: GameState):

    prompt = f"""
Create one coherent game concept from the following information.

ORIGINAL GAME IDEA:
{state["game_idea"]}

STORY:
{state["story"]}

CHARACTERS:
{state["characters"]}

GAMEPLAY:
{state["gameplay"]}

Previous feedback:
{" ".join(state["feedback"]) if state["feedback"] else "None"}

Combine these ideas into a single consistent game.

Return:

GAME TITLE:
...

STORY:
...

CHARACTERS:
...

GAMEPLAY:
...

OVERALL CONCEPT:
...
"""

    response = get_agent_response(aggregator_agent, prompt)

    print("\n========== AGGREGATED GAME CONCEPT ==========")
    print(response)

    return {
        "aggregated": response
    }

In [ ]:
def voting(state: GameState):

    concept = state["aggregated"]

    prompt = f"""
Review this game concept:

{concept}

Vote YES if you think the concept should be approved.

Vote NO if you think it needs improvement.
"""

    print("\n========== VOTING ==========")

    result_1 = get_agent_response(voter_1, prompt)
    result_2 = get_agent_response(voter_2, prompt)
    result_3 = get_agent_response(voter_3, prompt)

    print("\nVOTER 1:")
    print(result_1)

    print("\nVOTER 2:")
    print(result_2)

    print("\nVOTER 3:")
    print(result_3)

    return {
        "votes": [
            result_1,
            result_2,
            result_3
        ]
    }

In [ ]:
def extract_vote(text):

    text = text.upper()

    if "VOTE: YES" in text:
        return "YES"

    return "NO"

In [ ]:
def decide(state: GameState):

    votes = state["votes"][-3:]

    extracted_votes = [
        extract_vote(vote)
        for vote in votes
    ]

    yes_count = extracted_votes.count("YES")
    no_count = extracted_votes.count("NO")

    print("\n========== VOTE RESULT ==========")
    print("Votes:", extracted_votes)
    print("YES:", yes_count)
    print("NO:", no_count)

    if yes_count >= 2:
        return "approved"

    return "rejected"

In [ ]:
def feedback_node(state: GameState):

    latest_votes = state["votes"][-3:]

    no_feedback = []

    for vote in latest_votes:

        if extract_vote(vote) == "NO":
            no_feedback.append(vote)

    print("\n========== FEEDBACK ==========")

    for feedback in no_feedback:
        print(feedback)

    return {
        "feedback": no_feedback,
        "round": state["round"] + 1
    }

In [ ]:
graph = StateGraph(GameState)

In [ ]:
graph.add_node("orchestrator", orchestrator)

graph.add_node("story", story_node)
graph.add_node("characters", character_node)
graph.add_node("gameplay", gameplay_node)

graph.add_node("aggregator", aggregator)

graph.add_node("voting", voting)

graph.add_node("feedback", feedback_node)

In [ ]:
graph.add_edge(START, "orchestrator")

graph.add_edge("orchestrator", "story")
graph.add_edge("orchestrator", "characters")
graph.add_edge("orchestrator", "gameplay")

In [ ]:
graph.add_edge("story", "aggregator")
graph.add_edge("characters", "aggregator")
graph.add_edge("gameplay", "aggregator")

In [ ]:
graph.add_edge("aggregator", "voting")

In [ ]:
graph.add_conditional_edges(
    "voting",
    decide,
    {
        "approved": END,
        "rejected": "feedback"
    }
)

In [ ]:
graph.add_edge("feedback", "orchestrator")

In [ ]:
app = graph.compile()

In [ ]:
app.get_graph().draw_mermaid()

'---\nconfig:\n  flowchart:\n    curve: linear\n---\ngraph TD;\n\t__start__([<p>__start__</p>]):::first\n\torchestrator(orchestrator)\n\tstory(story)\n\tcharacters(characters)\n\tgameplay(gameplay)\n\taggregator(aggregator)\n\tvoting(voting)\n\tfeedback(feedback)\n\t__end__([<p>__end__</p>]):::last\n\t__start__ --> orchestrator;\n\taggregator --> voting;\n\tcharacters --> aggregator;\n\tfeedback --> orchestrator;\n\tgameplay --> aggregator;\n\torchestrator --> characters;\n\torchestrator --> gameplay;\n\torchestrator --> story;\n\tstory --> aggregator;\n\tvoting -. &nbsp;approved&nbsp; .-> __end__;\n\tvoting -. &nbsp;rejected&nbsp; .-> feedback;\n\tclassDef default fill:#f2f0ff,line-height:1.2\n\tclassDef first fill-opacity:0\n\tclassDef last fill:#bfb6fc\n'

In [ ]:
game_idea = """
A horror game where the player explores an abandoned hotel
in Egypt and discovers that the hotel is connected to a
mysterious disappearance that happened 30 years ago.
"""

In [ ]:
result = app.invoke({
    "game_idea": game_idea,
    "story": "",
    "characters": "",
    "gameplay": "",
    "aggregated": "",
    "votes": [],
    "feedback": [],
    "round": 1
})


========== ROUND 1 ==========

ORCHESTRATOR
Starting game concept discussion...

This is the first discussion round.

--- GAMEPLAY AGENT ---
### Gameplay Analysis for the Horror Game Set in an Abandoned Hotel

#### Core Gameplay Loop
1. **Exploration**: The player navigates through the abandoned hotel, uncovering different rooms and areas, each with its own atmosphere and lore.
2. **Discovery**: As players explore, they find clues (notes, photographs, audio recordings) that piece together the story of the mysterious disappearance.
3. **Puzzle Solving**: Players encounter environmental puzzles that must be solved to unlock new areas or progress the story.
4. **Survival Elements**: The player must manage resources (such as limited light sources, health items, and inventory space) while avoiding dangers within the hotel.
5. **Unraveling the Mystery**: Players connect clues and solve puzzles to uncover the truth behind the disappearance, leading to multiple possible endings based on their